# 07 — Testing with pytest

`🔴 Advanced`

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MusaIslamFahad/python-for-ai-engineers/blob/main/03_Advanced/07_testing_with_pytest.ipynb)

## 📌 What is it?
pytest is Python's most popular testing framework — simpler and more powerful than the built-in `unittest`.

## 🧠 Why AI Engineers need this
ML pipelines break silently. Tests catch regressions in preprocessing, model inputs, output parsing, and API clients before they hit production.

In [ ]:
# ── BASIC TESTS ──
# File: test_my_module.py
# Run with: pytest test_my_module.py -v

def chunk_text(text: str, size: int = 100) -> list[str]:
    return [text[i:i+size] for i in range(0, len(text), size)]

# Test functions start with test_
def test_chunk_text_basic():
    result = chunk_text("hello world", size=5)
    assert result == ["hello", " worl", "d"]

def test_chunk_text_exact_size():
    text = "A" * 10
    result = chunk_text(text, size=10)
    assert len(result) == 1

def test_chunk_text_empty():
    result = chunk_text("", size=10)
    assert result == []

# Run these locally with: pytest -v
# Let's simulate running:
for test in [test_chunk_text_basic, test_chunk_text_exact_size, test_chunk_text_empty]:
    try:
        test()
        print(f"✅ {test.__name__} PASSED")
    except AssertionError as e:
        print(f"❌ {test.__name__} FAILED: {e}")

In [ ]:
# ── FIXTURES ──
# Fixtures provide reusable test setup

import pytest   # run in real pytest environment

# @pytest.fixture
def sample_config():
    """Reusable test config."""
    return {
        "model": "gpt-4o",
        "temperature": 0.7,
        "max_tokens": 100
    }

# @pytest.fixture
def sample_documents():
    return [
        "Python is a programming language.",
        "Machine learning is a subset of AI.",
        "LLMs are trained on large text corpora."
    ]

# In a real pytest file, use @pytest.fixture and inject:
# def test_something(sample_config, sample_documents):
#     ...

# Demonstrate manually
cfg = sample_config()
docs = sample_documents()
print(f"Config model: {cfg['model']}")
print(f"Docs count: {len(docs)}")

In [ ]:
# ── PARAMETRIZE — run same test with multiple inputs ──
import pytest

def is_valid_temperature(t: float) -> bool:
    return 0.0 <= t <= 2.0

# In pytest:
# @pytest.mark.parametrize("temp,expected", [
#     (0.0, True),
#     (1.0, True),
#     (2.0, True),
#     (-0.1, False),
#     (2.1, False),
# ])
# def test_temperature(temp, expected):
#     assert is_valid_temperature(temp) == expected

# Simulate parametrized testing:
test_cases = [(0.0, True), (1.0, True), (2.0, True), (-0.1, False), (2.1, False)]
for temp, expected in test_cases:
    result = is_valid_temperature(temp)
    status = "✅" if result == expected else "❌"
    print(f"{status} temperature={temp} → {result} (expected {expected})")

In [ ]:
# ── MOCKING external calls ──
from unittest.mock import Mock, patch, MagicMock

# Mock an API call so tests don't hit real endpoints
def call_llm_api(prompt: str, client) -> str:
    response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

# In tests, mock the client
mock_client = MagicMock()
mock_client.messages.create.return_value = MagicMock(
    content=[MagicMock(text="Mocked response")]
)

result = call_llm_api("Hello!", mock_client)
print(f"Result: {result}")                                # Mocked response
mock_client.messages.create.assert_called_once()          # verify it was called
print("✅ Mock API test passed")

## ⚠️ Common Mistakes
```python
# ❌ Testing implementation details
def test_internal():
    assert model._hidden_state == some_value  # brittle!

# ✅ Test behavior/outputs
def test_behavior():
    output = model.predict(input)
    assert output.shape == expected_shape

# ❌ Not isolating tests — shared state between tests
# ✅ Use fixtures to create fresh state for each test

# Run tests:
# pytest tests/ -v              # verbose
# pytest tests/ --cov=src       # with coverage
# pytest tests/ -x              # stop on first failure
```

## 🔗 What's Next?
[04 Data Science — NumPy →](../04_Data_Science_Stack/01_numpy.ipynb)